In [ ]:
!pip install -U scikeras scikit-learn


In [ ]:
# Cell 1 - Load Dataset
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)

# Flatten images 28x28 -> 784
X_train = X_train.reshape(X_train.shape[0], 784).astype("float32") / 255
X_test = X_test.reshape(X_test.shape[0], 784).astype("float32") / 255

# One-hot encode labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


Training data shape: (60000, 28, 28)
Test data shape: (10000, 28, 28)
X_train shape: (60000, 784)
y_train shape: (60000, 10)


In [ ]:
# Cell 2 - Build Model Function
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam

def create_model(learning_rate=0.01):
    model = Sequential()
    model.add(Dense(64, input_dim=784, activation='relu'))
    model.add(Dense(10, activation='softmax'))

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model


In [ ]:
# Cell 3 - Grid Search with scikeras
!pip install scikeras

from sklearn.model_selection import GridSearchCV
from scikeras.wrappers import KerasClassifier

# Wrap model
model = KerasClassifier(model=create_model, verbose=0)

# Parameter grid (لاحظي model__learning_rate)
param_grid = {
    'batch_size': [32, 64],
    'epochs': [10],
    'model__learning_rate': [0.001, 0.01, 0.1]
}

grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3)
grid_result = grid.fit(X_train, y_train)

print("Best GridSearch params:", grid_result.best_params_)
print("Best GridSearch score:", grid_result.best_score_)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/lo

Best GridSearch params: {'batch_size': 32, 'epochs': 10, 'model__learning_rate': 0.001}
Best GridSearch score: 0.9686166666666667


In [ ]:
# Cell 4 - Random Search with scikeras
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

param_dist = {
    'batch_size': randint(16, 129),
    'epochs': randint(5, 20),
    'model__learning_rate': uniform(1e-5, 1e-1)
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    random_state=42
)

random_result = random_search.fit(X_train, y_train)

print("Best RandomSearch params:", random_result.best_params_)
print("Best RandomSearch score:", random_result.best_score_)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/lo

Best RandomSearch params: {'batch_size': 118, 'epochs': 14, 'model__learning_rate': 0.015609452033620266}
Best RandomSearch score: 0.9585666666666666


In [ ]:
# Cell 5 - Compare CV Results
import pandas as pd

grid_best_params = grid_result.best_params_
grid_best_score = grid_result.best_score_

random_best_params = random_result.best_params_
random_best_score = random_result.best_score_

comparison_df = pd.DataFrame([
    {"Method": "GridSearch", "Best Score": grid_best_score, "Best Params": grid_best_params},
    {"Method": "RandomSearch", "Best Score": random_best_score, "Best Params": random_best_params}
])

print(comparison_df)

better = "GridSearch" if grid_best_score > random_best_score else "RandomSearch"
print(f"\n🔥 الأفضل بناءً على CV Score هو: {better}")


         Method  Best Score                                        Best Params
0    GridSearch    0.968617  {'batch_size': 32, 'epochs': 10, 'model__learn...
1  RandomSearch    0.958567  {'batch_size': 118, 'epochs': 14, 'model__lear...

🔥 الأفضل بناءً على CV Score هو: GridSearch


In [ ]:
# Cell 6 - Final Model using Best Params (Grid or Random)
from sklearn.metrics import accuracy_score

# اختار الأفضل من حيث score
if grid_best_score >= random_best_score:
    best_params = grid_best_params
    print("✅ Using GridSearch best parameters")
else:
    best_params = random_best_params
    print("✅ Using RandomSearch best parameters")

# بناء الموديل بالـ best params
final_model = create_model(learning_rate=best_params['model__learning_rate'])

final_model.fit(X_train,
                y_train,
                epochs=best_params['epochs'],
                batch_size=best_params['batch_size'],
                verbose=1)

# تقييم على test set
final_preds = final_model.predict(X_test, verbose=0)
final_preds_classes = final_preds.argmax(axis=1)
y_test_classes = y_test.argmax(axis=1)

final_test_acc = accuracy_score(y_test_classes, final_preds_classes)
print(f"\n🎯 Final Test Accuracy: {final_test_acc:.4f}")


✅ Using GridSearch best parameters


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.8510 - loss: 0.5174
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9533 - loss: 0.1633
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9681 - loss: 0.1094
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9740 - loss: 0.0825
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9791 - loss: 0.0678
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9831 - loss: 0.0550
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9861 - loss: 0.0474
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9885 - loss: 0.0382
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9898 - loss: 0.0339
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9913 - loss: 0.0285

🎯 Final Test Accuracy: 0.9725
